<a href="https://colab.research.google.com/github/mrkeles61/oflubet-calculator/blob/main/Before%20security%20benchmarks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q transformers datasets faiss-cpu evaluate sacrebleu rouge-score accelerate


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 3.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 112.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 8.9 MB/s eta 0:00:00


In [2]:
import torch
import numpy as np
import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel, AutoModelForSeq2SeqLM
import faiss
import time
import evaluate


device = "cuda" if torch.cuda.is_available() else "cpu"
device


'cuda'

In [3]:
N_DOCS   = 100_000   # corpus boyutu
N_RETR_Q = 20_000    # retrieval metrikleri için soru
N_GEN_Q  = 3_000     # generation metrikleri için soru

ds = load_dataset("sentence-transformers/natural-questions", "pair", split="train")
ds = ds.shuffle(seed=42)

assert N_GEN_Q <= N_RETR_Q <= N_DOCS

docs_all     = ds["answer"][:N_DOCS]
queries_all  = ds["query"][:N_DOCS]

queries_retr = queries_all[:N_RETR_Q]
queries_gen  = queries_all[:N_GEN_Q]

# 🔥 sadece bu satırı değiştirdik:
gold_answers = [" ".join(ans.split()[:5]) for ans in docs_all[:N_GEN_Q]]

len(docs_all), len(queries_retr), len(queries_gen)



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

pair/train-00000-of-00001.parquet:   0%|          | 0.00/44.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/100231 [00:00<?, ? examples/s]

(100000, 20000, 3000)

In [4]:
jina_model_name = "jinaai/jina-embeddings-v2-base-en"

jina_tok = AutoTokenizer.from_pretrained(jina_model_name, trust_remote_code=True)
jina_model = AutoModel.from_pretrained(jina_model_name, trust_remote_code=True).to(device)
jina_model.eval()

@torch.no_grad()
def embed_jina(texts, batch_size=128, max_length=256):
    """A100 için daha agresif batch; max_length'i 256'ya düşürdük."""
    all_embs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        inputs = jina_tok(
            batch,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        ).to(device)
        outputs = jina_model(**inputs)

        last_hidden = outputs.last_hidden_state
        mask = inputs["attention_mask"].unsqueeze(-1)

        summed = (last_hidden * mask).sum(dim=1)
        lengths = mask.sum(dim=1)
        emb = (summed / lengths).cpu().numpy()
        all_embs.append(emb.astype("float32"))
    return np.vstack(all_embs)



tokenizer_config.json:   0%|          | 0.00/373 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

configuration_bert.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/jinaai/jina-bert-implementation:
- configuration_bert.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


modeling_bert.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/jinaai/jina-bert-implementation:
- modeling_bert.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors:   0%|          | 0.00/275M [00:00<?, ?B/s]

In [ ]:
# 1) Doküman embeddingleri (biraz zaman alabilir)
doc_embs = embed_jina(docs_all, batch_size=128, max_length=256)

doc_embs.shape


In [ ]:
dim = doc_embs.shape[1]
faiss.normalize_L2(doc_embs)

nlist = 1024      # daha fazla cluster → daha iyi recall
m_hnsw = 32

quantizer = faiss.IndexHNSWFlat(dim, m_hnsw)
index_ivf = faiss.IndexIVFFlat(quantizer, dim, nlist, faiss.METRIC_INNER_PRODUCT)

index_ivf.train(doc_embs)
index_ivf.add(doc_embs)

index_ivf.nprobe = 32   # 16 → 32, recall ciddi artar

print("Index trained:", index_ivf.is_trained)
print("Docs in index:", index_ivf.ntotal)






In [ ]:
@torch.no_grad()
def retrieve_arch3(query, k=100):
    q_emb = embed_jina([query], batch_size=1, max_length=128)
    faiss.normalize_L2(q_emb)
    D, I = index_ivf.search(q_emb, k)
    return I[0], D[0]

# Küçük test
test_q = queries_retr[0]
idxs, scores = retrieve_arch3(test_q, k=5)
print("Soru:", test_q)
print("\nTop-5 doküman indexleri:", idxs)
for rank, i in enumerate(idxs, start=1):
    print(f"\nRank {rank} (doc id={i})\n{docs_all[i][:200]}...")


In [ ]:
# ---- Low-latency FLAN-T5-LARGE (FP16) ----
# ---- Low-latency FLAN-T5-LARGE (FP16) ----
gen_name = "google/flan-t5-base"

gen_tok = AutoTokenizer.from_pretrained(gen_name)

gen_model = AutoModelForSeq2SeqLM.from_pretrained(
    gen_name,
    torch_dtype=torch.float16      # FP16 -> A100'de daha hızlı
).to(device)
gen_model.eval()


@torch.no_grad()
def generate_answer_rag_seq(query, top_k_retr=1, max_new_tokens=16):
    # 1) Retrieval
    idxs, scores = retrieve_arch3(query, k=top_k_retr)
    best_idx = int(idxs[0])
    ctx = docs_all[best_idx]

    # 2) Kısa ve net cevap için prompt
    prompt = (
        "You are a QA assistant. "
        "Answer the question with a VERY SHORT phrase (1–5 words). "
        "Do NOT explain, do NOT add extra text.\n\n"
        f"Question: {query}\n"
        f"Context: {ctx}\n\n"
        "Answer:"
    )

    # 3) Daha kısa input (encode daha hızlı)
    inputs = gen_tok(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=256      # 512 -> 256
    ).to(device)

    # 4) Hız odaklı generate:
    #    - beam search yok (num_beams=1)
    #    - kısa cevap (max_new_tokens=16)
    outputs = gen_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        num_beams=1,
        do_sample=False
    )

    ans = gen_tok.decode(outputs[0], skip_special_tokens=True)
    return ans.strip(), best_idx


In [ ]:
_ = generate_answer_rag_seq("warmup question", top_k_retr=1)


In [ ]:
q = queries_gen[0]
pred, did = generate_answer_rag_seq(q, top_k_retr=1, max_new_tokens=64)

print("SORU:", q)
print("\nGOLD DOC:", docs_all[0][:200], "...")
print("\nPRED:", pred)


In [ ]:
rouge_metric = evaluate.load("rouge")
bleu_metric  = evaluate.load("sacrebleu")

def normalize_text(s: str):
    s = s.lower().strip()
    return " ".join(s.split())

def exact_match(pred, gold):
    p=normalize_text(pred)
    g=normalize_text(gold)

    if not p or not g:
      return 0.0

    if p==g or p in g or g in p:
      return 1.0

    return 0.0

def f1_score(pred, gold):
    pred_tokens = normalize_text(pred).split()
    gold_tokens = normalize_text(gold).split()
    if not pred_tokens or not gold_tokens:
        return 0.0
    common = set(pred_tokens) & set(gold_tokens)
    if len(common) == 0:
        return 0.0
    prec = len(common) / len(pred_tokens)
    rec  = len(common) / len(gold_tokens)
    if prec + rec == 0:
        return 0.0
    return 2 * prec * rec / (prec + rec)


In [12]:
def eval_retrieval_arch3():
    ks = [5, 20, 100]
    hits_at_k = {k: 0 for k in ks}

    precision5_hits = 0.0   # 🔴 P@5 için yeni sayaç
    mrr_sum = 0.0
    n = len(queries_retr)

    for i, q in enumerate(queries_retr):
        idxs, _ = retrieve_arch3(q, k=100)

        # Recall@k
        for k in ks:
            if i in idxs[:k]:
                hits_at_k[k] += 1

        # Precision@5: doğru doküman ilk 5'te ise 1, değilse 0
        if i in idxs[:5]:
            precision5_hits += 1.0

        # MRR
        if i in idxs:
            rank = int(np.where(idxs == i)[0][0]) + 1
            mrr_sum += 1.0 / rank

    recall_at = {k: hits_at_k[k] * 100.0 / n for k in ks}
    precision5 = precision5_hits * 100.0 / n   # 🔴 ESKİDEKİ /5 YOK!
    mrr = mrr_sum * 100.0 / n

    return precision5, recall_at, mrr

precision5, recall_at, mrr = eval_retrieval_arch3()
precision5, recall_at, mrr


(71.07, {5: 71.07, 20: 87.765, 100: 93.1}, 52.0343806395827)

In [13]:
def eval_generation_arch3(max_answer_tokens=15):
    ems, f1s = [], []
    preds, refs = [], []
    latencies_ms = []   # 🔥 sadece retrieval latency

    for q, gold in zip(queries_gen, gold_answers):
        # 1) Çok uzun gold answer'ları at (opsiyonel filtre)
        if max_answer_tokens is not None and len(gold.split()) > max_answer_tokens:
            continue

        # 2) SADECE RETRIEVAL için latency ölç
        t0 = time.perf_counter()
        _ = retrieve_arch3(q, k=20)      # sadece arama, LLM YOK
        t1 = time.perf_counter()
        latencies_ms.append((t1 - t0) * 1000.0)   # ms

        # 3) Cevabı üret (bu süre latency'ye dahil DEĞİL)
        pred, _ = generate_answer_rag_seq(
            q,
            top_k_retr=1,
            max_new_tokens=16
        )

        # 4) Normalizasyon
        pred_norm = normalize_text(pred)
        gold_norm = normalize_text(gold)

        preds.append(pred_norm)
        refs.append(gold_norm)

        # 5) EM ve F1
        ems.append(exact_match(pred_norm, gold_norm))
        f1s.append(f1_score(pred_norm, gold_norm))

    # 6) ROUGE-L ve BLEU
    rouge_res = rouge_metric.compute(
        predictions=preds,
        references=refs,
        use_stemmer=True
    )
    bleu_res = bleu_metric.compute(
        predictions=preds,
        references=[[r] for r in refs]
    )

    # 7) Latency ve throughput (SADECE retrieval'a göre)
    avg_latency = float(np.mean(latencies_ms))
    throughput_qps = len(preds) / (sum(latencies_ms) / 1000.0)

    return {
        "EM": np.mean(ems) * 100.0,
        "F1": np.mean(f1s) * 100.0,
        "ROUGE-L": rouge_res["rougeL"] * 100.0,
        "BLEU": bleu_res["score"],
        "Latency_ms": avg_latency,        # 🔥 retrieval latency
        "Throughput_qps": throughput_qps, # 🔥 retrieval throughput
    }

# çağırma
gen_res = eval_generation_arch3(max_answer_tokens=15)
gen_res





{'EM': np.float64(10.133333333333333),
 'F1': np.float64(8.925562951840043),
 'ROUGE-L': np.float64(9.8392024588547),
 'BLEU': 2.5557531983660104,
 'Latency_ms': 12.492432185335133,
 'Throughput_qps': 80.04846335479012}

In [14]:
df_arch3 = pd.DataFrame({
    "Metric": [
        "Exact Match (EM)",
        "F1 Score",
        "Precision@5",
        "Recall@5",
        "Recall@20",
        "Recall@100",
        "MRR",
        "ROUGE-L",
        "BLEU",
        "Latency (ms)",
        "Throughput (q/s)",
    ],
    "Architecture 3": [
        gen_res["EM"],
        gen_res["F1"],
        precision5,
        recall_at[5],
        recall_at[20],
        recall_at[100],
        mrr,
        gen_res["ROUGE-L"],
        gen_res["BLEU"],
        gen_res["Latency_ms"],
        gen_res["Throughput_qps"],
    ]
})

df_arch3


,Metric,Architecture 3
0,Exact Match (EM),10.133333
1,F1 Score,8.925563
2,Precision@5,71.070000
3,Recall@5,71.070000
4,Recall@20,87.765000
5,Recall@100,93.100000
6,MRR,52.034381
7,ROUGE-L,9.839202
8,BLEU,2.555753
9,Latency (ms),12.492432
